In [4]:
# !pip install -q pandas openpyxl

import pandas as pd
import re
import html
import shutil
from openpyxl import load_workbook
from openpyxl.styles import Font

# ============================================================
# FILE PATHS
# ============================================================

main_file = "/content/franckenachlass_6.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"
corrected_file = "/content/organization_updated.xlsx"
output_file = "/content/franckenachlass_7.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"

main_sheet = "Sheet1"
main_col = "subject_corporate_610"

# ============================================================
# CORRECTED FILE COLUMN NAMES
# ============================================================

wikidata_url_col = "Wikidata URL"
wikidata_label_col = "Wikidata Label"

lcnaf_url_col = "LCNAF URL"
lcnaf_label_col = "LCNAF Label"

viaf_url_col = "VIAF URL"
viaf_label_col = "VIAF Label"

# ============================================================
# HELPERS
# ============================================================

def is_blank(value):
    return (
        value is None
        or pd.isna(value)
        or str(value).strip() == ""
    )


def normalize_colname(value):
    value = str(value)
    value = value.replace("\ufeff", "")
    value = value.replace("\xa0", " ")
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def find_column(df, target):
    target_norm = normalize_colname(target).lower()

    for col in df.columns:
        if normalize_colname(col).lower() == target_norm:
            return col

    return None


def normalize_label(value):
    """
    Used only to compare labels after a URI match has already been found.
    Labels are NEVER used to identify a corrected record.
    """

    if is_blank(value):
        return ""

    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text.lower()


# ============================================================
# URI / AUTHORITY ID EXTRACTION
# ============================================================

def get_wikidata_id(value):
    if is_blank(value):
        return ""

    m = re.search(
        r"\bQ\d+\b",
        str(value),
        flags=re.IGNORECASE
    )

    return m.group(0).upper() if m else ""


def get_lcnaf_id(value):
    if is_blank(value):
        return ""

    text = str(value).strip()

    # Full URI
    m = re.search(
        r"/authorities/names/([A-Za-z0-9]+)",
        text,
        flags=re.IGNORECASE
    )

    if m:
        return m.group(1).lower()

    # Bare ID
    if re.fullmatch(
        r"[A-Za-z]{1,4}\d+[A-Za-z0-9]*",
        text
    ):
        return text.lower()

    return ""


def get_viaf_id(value):
    if is_blank(value):
        return ""

    text = str(value).strip()

    # Full URI
    m = re.search(
        r"/viaf/(\d+)",
        text,
        flags=re.IGNORECASE
    )

    if m:
        return m.group(1)

    # Bare numeric ID
    if text.isdigit():
        return text

    return ""


def authority_id(uri, source):
    """
    Returns the stable authority identifier.

    Examples:
        LC       -> n82166282
        VIAF     -> 139236953
        Wikidata -> Q13417577
    """

    source = source.upper()

    if source == "LC":
        return get_lcnaf_id(uri)

    if source == "VIAF":
        return get_viaf_id(uri)

    if source == "WIKIDATA":
        return get_wikidata_id(uri)

    return ""


# ============================================================
# HTML PARSER FOR AUTHORITY LINKS
# ============================================================

authority_pattern = re.compile(
    r'(?P<prefix>'
    r'<p[^>]*>\s*'
    r'<a[^>]*href=["\']'
    r')'
    r'(?P<uri>[^"\']+)'
    r'(?P<middle>'
    r'["\'][^>]*>'
    r')'
    r'(?P<label>.*?)'
    r'(?P<suffix>'
    r'</a>\s*'
    r'\['
    r'(?P<source>LC|VIAF|Wikidata)'
    r'\]'
    r'\s*</p>'
    r')',
    flags=re.IGNORECASE | re.DOTALL
)


def parse_authority_entries(cell_value):
    if is_blank(cell_value):
        return []

    text = str(cell_value)

    entries = []

    for match in authority_pattern.finditer(text):

        source = match.group("source").strip().upper()

        if source == "WIKIDATA":
            source = "WIKIDATA"

        entries.append({
            "source": source,
            "uri": html.unescape(
                match.group("uri")
            ).strip(),
            "label": html.unescape(
                match.group("label")
            ).strip(),
            "match": match
        })

    return entries


# ============================================================
# READ CORRECTED FILE
# ============================================================

corrected_df = pd.read_excel(
    corrected_file
)

# ============================================================
# LOCATE CORRECTED FILE COLUMNS
# ============================================================

actual_wikidata_url_col = find_column(
    corrected_df,
    wikidata_url_col
)

actual_wikidata_label_col = find_column(
    corrected_df,
    wikidata_label_col
)

actual_lcnaf_url_col = find_column(
    corrected_df,
    lcnaf_url_col
)

actual_lcnaf_label_col = find_column(
    corrected_df,
    lcnaf_label_col
)

actual_viaf_url_col = find_column(
    corrected_df,
    viaf_url_col
)

actual_viaf_label_col = find_column(
    corrected_df,
    viaf_label_col
)

required_columns = {
    "Wikidata URL": actual_wikidata_url_col,
    "Wikidata Label": actual_wikidata_label_col,
    "LCNAF URL": actual_lcnaf_url_col,
    "LCNAF Label": actual_lcnaf_label_col,
    "VIAF URL": actual_viaf_url_col,
    "VIAF Label": actual_viaf_label_col
}

missing = [
    name
    for name, actual in required_columns.items()
    if actual is None
]

if missing:
    raise ValueError(
        "Missing corrected-file columns: "
        + ", ".join(missing)
    )

# ============================================================
# BUILD URI LOOKUP TABLES
# ============================================================

# Each dictionary is:
#
# authority_id -> {
#     "uri": corrected URI,
#     "label": corrected label,
#     "row": corrected Excel row
# }

lc_lookup = {}
viaf_lookup = {}
wikidata_lookup = {}


for df_index, row in corrected_df.iterrows():

    excel_row = df_index + 2

    # --------------------------------------------------------
    # LCNAF
    # --------------------------------------------------------

    lc_uri = row[actual_lcnaf_url_col]
    lc_label = row[actual_lcnaf_label_col]

    lc_id = authority_id(
        lc_uri,
        "LC"
    )

    if lc_id:

        lc_lookup[lc_id] = {
            "uri": str(lc_uri).strip(),
            "label": (
                ""
                if is_blank(lc_label)
                else str(lc_label).strip()
            ),
            "row": excel_row
        }

    # --------------------------------------------------------
    # VIAF
    # --------------------------------------------------------

    viaf_uri = row[actual_viaf_url_col]
    viaf_label = row[actual_viaf_label_col]

    viaf_id = authority_id(
        viaf_uri,
        "VIAF"
    )

    if viaf_id:

        viaf_lookup[viaf_id] = {
            "uri": str(viaf_uri).strip(),
            "label": (
                ""
                if is_blank(viaf_label)
                else str(viaf_label).strip()
            ),
            "row": excel_row
        }

    # --------------------------------------------------------
    # Wikidata
    # --------------------------------------------------------

    wd_uri = row[actual_wikidata_url_col]
    wd_label = row[actual_wikidata_label_col]

    wd_id = authority_id(
        wd_uri,
        "WIKIDATA"
    )

    if wd_id:

        wikidata_lookup[wd_id] = {
            "uri": str(wd_uri).strip(),
            "label": (
                ""
                if is_blank(wd_label)
                else str(wd_label).strip()
            ),
            "row": excel_row
        }


LOOKUPS = {
    "LC": lc_lookup,
    "VIAF": viaf_lookup,
    "WIKIDATA": wikidata_lookup
}


print("Authority records loaded:")

print(
    f"  LCNAF: {len(lc_lookup)}"
)

print(
    f"  VIAF: {len(viaf_lookup)}"
)

print(
    f"  Wikidata: {len(wikidata_lookup)}"
)

# ============================================================
# COPY ENTIRE ORIGINAL WORKBOOK
# ============================================================

shutil.copy2(
    main_file,
    output_file
)

# ============================================================
# OPEN COPIED WORKBOOK
# ============================================================

wb = load_workbook(
    output_file
)

if main_sheet not in wb.sheetnames:
    raise ValueError(
        f"Sheet '{main_sheet}' not found."
    )

ws = wb[main_sheet]

# ============================================================
# FIND subject_corporate_610 COLUMN
# ============================================================

main_column_number = None

for cell in ws[1]:

    if (
        normalize_colname(cell.value).lower()
        ==
        normalize_colname(main_col).lower()
    ):
        main_column_number = cell.column
        break


if main_column_number is None:
    raise ValueError(
        f"Could not find '{main_col}' "
        f"in sheet '{main_sheet}'."
    )

print(
    f"\nProcessing column: {main_col}"
)

# ============================================================
# LOGS
# ============================================================

replacement_logs = []
uncontrolled_logs = []

# ============================================================
# PROCESS EACH ROW
# ============================================================

for excel_row in range(
    2,
    ws.max_row + 1
):

    cell = ws.cell(
        row=excel_row,
        column=main_column_number
    )

    original_value = cell.value

    if is_blank(original_value):
        continue

    original_text = str(
        original_value
    )

    entries = parse_authority_entries(
        original_text
    )

    # --------------------------------------------------------
    # No recognizable authority entries
    # --------------------------------------------------------

    if not entries:

        uncontrolled_logs.append({
            "Excel_Row": excel_row,
            "Authority_Source": "",
            "Authority_ID": "",
            "Existing_URI": "",
            "Existing_Label": "",
            "Reason":
                "No recognizable LC/VIAF/Wikidata link found",
            "Action":
                "KEPT UNCHANGED"
        })

        continue

    replacements = []

    # ========================================================
    # PROCESS EVERY AUTHORITY ENTRY INDEPENDENTLY
    # ========================================================

    for entry in entries:

        source = entry["source"]
        current_uri = entry["uri"]
        current_label = entry["label"]

        current_id = authority_id(
            current_uri,
            source
        )

        lookup = LOOKUPS[source]

        # ----------------------------------------------------
        # Could not extract authority ID
        # ----------------------------------------------------

        if not current_id:

            uncontrolled_logs.append({
                "Excel_Row": excel_row,
                "Authority_Source": source,
                "Authority_ID": "",
                "Existing_URI": current_uri,
                "Existing_Label": current_label,
                "Reason":
                    "Could not extract authority ID from URI",
                "Action":
                    "KEPT UNCHANGED"
            })

            continue

        # ----------------------------------------------------
        # URI / authority ID NOT found in corrected file
        #
        # DO NOT use label fallback.
        # KEEP unchanged.
        # ----------------------------------------------------

        if current_id not in lookup:

            uncontrolled_logs.append({
                "Excel_Row": excel_row,
                "Authority_Source": source,
                "Authority_ID": current_id,
                "Existing_URI": current_uri,
                "Existing_Label": current_label,
                "Reason":
                    "URI/authority ID not found in corrected file",
                "Action":
                    "KEPT UNCHANGED"
            })

            continue

        # ----------------------------------------------------
        # URI successfully identified
        # ----------------------------------------------------

        corrected = lookup[
            current_id
        ]

        corrected_uri = corrected["uri"]
        corrected_label = corrected["label"]
        corrected_row = corrected["row"]

        # ----------------------------------------------------
        # IMPORTANT:
        #
        # Since the ID matches, we treat the URI as the same
        # authority even if:
        #     http vs https
        #     trailing slash
        #     VIAF /en/viaf/
        #     LC .html
        #
        # So we DO NOT replace the URI just for formatting.
        # ----------------------------------------------------

        # ----------------------------------------------------
        # If corrected label is blank, keep original unchanged
        # ----------------------------------------------------

        if is_blank(corrected_label):

            uncontrolled_logs.append({
                "Excel_Row": excel_row,
                "Authority_Source": source,
                "Authority_ID": current_id,
                "Existing_URI": current_uri,
                "Existing_Label": current_label,
                "Reason":
                    "URI found, but corrected label is blank",
                "Action":
                    "KEPT UNCHANGED"
            })

            continue

        # ----------------------------------------------------
        # Compare label
        # ----------------------------------------------------

        label_matches = (
            normalize_label(current_label)
            ==
            normalize_label(corrected_label)
        )

        # ----------------------------------------------------
        # URI found + label already correct
        # KEEP EXACTLY AS IT IS
        # ----------------------------------------------------

        if label_matches:
            continue

        # ----------------------------------------------------
        # URI found + label differs
        #
        # Replace ONLY label.
        # Keep original URI exactly as written.
        # ----------------------------------------------------

        match = entry["match"]

        new_html = (
            match.group("prefix")
            +
            html.escape(
                current_uri,
                quote=True
            )
            +
            match.group("middle")
            +
            html.escape(
                corrected_label,
                quote=False
            )
            +
            match.group("suffix")
        )

        replacements.append({
            "start":
                match.start(),
            "end":
                match.end(),
            "new_html":
                new_html
        })

        replacement_logs.append({
            "Excel_Row":
                excel_row,

            "Authority_Source":
                source,

            "Authority_ID":
                current_id,

            "Corrected_File_Row":
                corrected_row,

            "URI":
                current_uri,

            "Old_Label":
                current_label,

            "New_Label":
                corrected_label,

            "Changed":
                "LABEL"
        })

    # ========================================================
    # APPLY REPLACEMENTS RIGHT-TO-LEFT
    # ========================================================

    if replacements:

        updated_text = original_text

        for replacement in sorted(
            replacements,
            key=lambda x: x["start"],
            reverse=True
        ):

            updated_text = (
                updated_text[
                    :replacement["start"]
                ]
                +
                replacement["new_html"]
                +
                updated_text[
                    replacement["end"]:
                ]
            )

        cell.value = updated_text

# ============================================================
# REMOVE OLD LOG TABS ONLY
# ============================================================

for log_sheet in [
    "Org_Replacement_Log",
    "Org_Uncontrolled_Log"
]:

    if log_sheet in wb.sheetnames:
        del wb[log_sheet]

# ============================================================
# CREATE REPLACEMENT LOG
# ============================================================

replacement_ws = wb.create_sheet(
    "Org_Replacement_Log"
)

replacement_headers = [
    "Excel_Row",
    "Authority_Source",
    "Authority_ID",
    "Corrected_File_Row",
    "URI",
    "Old_Label",
    "New_Label",
    "Changed"
]

replacement_ws.append(
    replacement_headers
)

for item in replacement_logs:

    replacement_ws.append([
        item.get(
            header,
            ""
        )
        for header
        in replacement_headers
    ])

# ============================================================
# CREATE UNCONTROLLED LOG
# ============================================================

uncontrolled_ws = wb.create_sheet(
    "Org_Uncontrolled_Log"
)

uncontrolled_headers = [
    "Excel_Row",
    "Authority_Source",
    "Authority_ID",
    "Existing_URI",
    "Existing_Label",
    "Reason",
    "Action"
]

uncontrolled_ws.append(
    uncontrolled_headers
)

for item in uncontrolled_logs:

    uncontrolled_ws.append([
        item.get(
            header,
            ""
        )
        for header
        in uncontrolled_headers
    ])

# ============================================================
# FORMAT LOG SHEETS
# ============================================================

for log_ws in [
    replacement_ws,
    uncontrolled_ws
]:

    # Bold header
    for c in log_ws[1]:
        c.font = Font(
            bold=True
        )

    # Freeze header
    log_ws.freeze_panes = "A2"

    # Filter
    log_ws.auto_filter.ref = (
        log_ws.dimensions
    )

    # Widths
    for column_cells in log_ws.columns:

        column_letter = (
            column_cells[0]
            .column_letter
        )

        max_length = 0

        for c in column_cells:

            value = (
                ""
                if c.value is None
                else str(c.value)
            )

            max_length = max(
                max_length,
                min(
                    len(value),
                    80
                )
            )

        log_ws.column_dimensions[
            column_letter
        ].width = max(
            12,
            min(
                max_length + 2,
                80
            )
        )

# ============================================================
# SAVE
# ============================================================

wb.save(
    output_file
)

wb.close()

# ============================================================
# SUMMARY
# ============================================================

print("\nDone.")

print(
    f"Output saved as:\n{output_file}"
)

print(
    f"\nLabel replacements: "
    f"{len(replacement_logs)}"
)

print(
    f"Uncontrolled / URI-not-found entries: "
    f"{len(uncontrolled_logs)}"
)

print(
    "\nAll original workbook tabs were preserved."
)

print(
    "Only these log tabs were added/replaced:"
)

print(
    "  - Replacement_Log"
)

print(
    "  - Uncontrolled_Log"
)

Authority records loaded:
  LCNAF: 5
  VIAF: 7
  Wikidata: 52

Processing column: subject_corporate_610

Done.
Output saved as:
/content/franckenachlass_7.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx

Label replacements: 0
Uncontrolled / URI-not-found entries: 0

All original workbook tabs were preserved.
Only these log tabs were added/replaced:
  - Replacement_Log
  - Uncontrolled_Log
